# Tokenizer
We started to experiment with `Journey to the west`

In [223]:
import requests
import pandas as pd
from pathlib import Path
import json
import regex
import time

In [213]:
pat = regex.compile(
    r"[\p{P}\p{S}]|'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+[\p{L}&&\n^\p{Han}]++|[\p{Han}]{1,2}+|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s")

In [8]:
def get_locations(corpus):

    DATA = Path("../data")
    DATA.mkdir(exist_ok=True)
    TK_DATA = DATA / "tokenizer"
    TK_DATA.mkdir(exist_ok=True)
    CORPUS_DATA = TK_DATA / corpus
    CORPUS_DATA.mkdir(exist_ok=True)
    return CORPUS_DATA

In [ ]:
CORPUS = "xyj"
CORPUS_DATA = get_locations(CORPUS)

In [ ]:
if CORPUS =="xyj":
    url = "https://raw.githubusercontent.com/tennessine/corpus/refs/heads/master/%E8%A5%BF%E6%B8%B8%E8%AE%B0.txt"

In [13]:
def get_text(url):
    if (CORPUS_DATA / "corpus.txt").exists() is False:
        text = requests.get(url).text
        with open(CORPUS_DATA / "corpus.txt", "w") as f:
            f.write(text)
    else:
        text = (CORPUS_DATA / "corpus.txt").read_text()

    return text

In [14]:
text = get_text(url)

In [15]:
text[:100]

'《西游记》吴承恩（明）\n\n第一回 灵根育孕源流出 心性修持大道生\n\n    诗曰：\n\n    混沌未分天地乱茫茫渺渺无人见。\n\n    自从盘古破鸿蒙开辟从兹清浊辨。\n\n    覆载群生仰至仁明万物皆'

In [5]:
len(text)

684043

In [6]:
utf8 = list(text.encode())

In [7]:
len(utf8)

1999157

In [8]:
freq = dict()

for c1, c2 in zip(utf8,utf8[1:]):
    freq[(c1, c2)] = freq.get((c1, c2),0) + 1

In [9]:
most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])
most_freq[:5]

[((228, 184), 44120),
 ((227, 128), 23200),
 ((239, 188), 22848),
 ((226, 128), 22574),
 ((128, 130), 21167)]

In [255]:
import time

In [256]:
# code_list = list(utf8)

code_collection = list(list(sub_text.encode("utf8")) for sub_text in pat.findall(text))

codes_start = 0
utf_len = len(list(text.encode("utf8")))
for l in code_collection:

    codes_start = max(max(l), codes_start)

vocab = dict()


def merge(ids, temp_vocab):
    if len(ids) < 2:
        return ids
    new_ids = []
    i = 0
    while i < len(ids):
        if i == len(ids) - 1:
            new_ids.append(ids[i])
            break
        token = temp_vocab.get((ids[i], ids[i + 1]))
        if token is not None:
            new_ids.append(token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

logs = []

last_ts = time.time()
while True:
    freq = dict()

    for code_list in code_collection:

        if len(code_list) < 2:
            continue

        for c1, c2 in zip(code_list, code_list[1:]):
            freq[(c1, c2)] = freq.get((c1, c2), 0) + 1

    most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])

    temp_vocab = dict()
    top_freq_ids1 = set()
    top_freq_ids2 = set()

    for (c1, c2), ct in most_freq:
        if c1 in top_freq_ids2 or c2 in top_freq_ids1 or ct < 5:
            break
        top_freq_ids1.add(c1)
        top_freq_ids2.add(c2)

        codes_start += 1

        temp_vocab[(c1, c2)] = codes_start

        vocab[codes_start] = (c1, c2)

    # print(temp_vocab)

    if len(temp_vocab) == 0:
        print(f"\nFinished Vocab:{len(vocab)} /{ct} / {ct/utf_len:.5f}")
        break

    # print(f"Try to merge {len(temp_vocab)}")
    new_code = codes_start

    if len(vocab) % 100 == 0:
        print(f"V:{len(vocab)}x", end="\t")

    ct = 0

    for i in range(len(code_collection)):
        code_collection[i] = merge(code_collection[i], temp_vocab)
        ct += len(code_collection[i])

    new_ts = time.time()

    logs.append({
        "vocab_len": len(vocab),
        "code_list_len": ct,
        "compression": ct / utf_len,
        "time": new_ts - last_ts,
    })

    last_ts = new_ts
    

V:3900x	V:4300x	V:4500x	V:4600x	V:6400x	V:8500x	V:10700x	V:11000x	V:11800x	V:13300x	V:14100x	V:14400x	
Finished Vocab:15534 /4 / 0.00000


In [257]:
df = pd.DataFrame(logs)
df

,vocab_len,code_list_len,compression,time
0,4,1879793,0.943418,0.566682
1,14,1744669,0.875603,0.594832
2,28,1620839,0.813456,0.500217
3,44,1518235,0.761962,0.471404
4,60,1436351,0.720866,0.448703
...,...,...,...,...
781,15471,430327,0.215970,0.227692
782,15472,430322,0.215967,0.242824
783,15476,430302,0.215957,0.249224
784,15507,430147,0.215879,0.241469


In [258]:
with open(CORPUS_DATA / "vocab.json", "w") as f:
    f.write(json.dumps(vocab, indent=2), )

with open(CORPUS_DATA / "freq.json", "w") as f:
    f.write(json.dumps(most_freq, indent=2))

In [259]:
df = pd.DataFrame(logs)
df.to_csv(str(CORPUS_DATA / "logs.csv"), index=False)

## Retro Analysis

In [260]:
import json
import pandas as pd

In [261]:
df = pd.read_csv(str(CORPUS_DATA / "logs.csv"))

In [262]:
with open(CORPUS_DATA/"vocab.json") as f:
    vocab = dict((int(k), v) for k, v in json.loads(f.read()).items())

with open(CORPUS_DATA/"freq.json") as f:
    most_freq = json.loads(f.read())

In [263]:
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_df(df):
    cols = ["vocab_len", "code_list_len", "compression", "time"]
    titles = ["Vocab Size", "Code List Length", "Compression Ratio", "Time per Step (s)"]

    fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

    for i, (col, title) in enumerate(zip(cols, titles)):
        row, col_idx = divmod(i, 2)
        fig.add_trace(
            go.Scatter(x=df.index, y=df[col], mode="lines", name=title),
            row=row + 1, col=col_idx + 1,
        )

    fig.update_layout(height=600, title_text="BPE Tokenizer Training", showlegend=False)
    fig.show()


In [264]:
plot_df(df)

In [265]:
pair_to_code = dict((tuple(v), int(k)) for k, v in vocab.items())

In [266]:
from loguru import logger

In [267]:
def encode(text: str, pair_to_code):
    code_list = list(text.encode("utf8"))
    if len(code_list) < 2:
        return code_list
    while len(code_list) > 1:
        temp_vocab = dict()
        for c1, c2 in zip(code_list, code_list[1:]):
            token = pair_to_code.get((c1, c2))
            if token is None:
                continue
            else:
                temp_vocab[(c1, c2)] = token
        if len(temp_vocab) == 0:
            return code_list


        pair = min(temp_vocab, key=lambda x: temp_vocab[x])
        token = temp_vocab[pair]

        new_code_list = []

        i = 0
        while i < len(code_list):
            if i < len(code_list) - 1 and code_list[i] == pair[0] and code_list[i + 1] == pair[1]:
                logger.info(f"[REPLACE]{code_list[i]}, {code_list[i + 1]} -> {token}")
                new_code_list.append(token)
                i += 2
            else:
                new_code_list.append(code_list[i])
                i += 1
        
        code_list = new_code_list
            
    return code_list

In [268]:
list("东土大唐".encode("utf8"))

[228, 184, 156, 229, 156, 159, 229, 164, 167, 229, 148, 144]

In [269]:
token_ids = encode("东土大唐", pair_to_code=pair_to_code)
print(token_ids)

[2109]


In [270]:
token_ids = encode("贫僧来自东土大唐", pair_to_code=pair_to_code)
print(token_ids)

[1243, 281, 527, 2109]


In [272]:
token_ids = encode("东土大唐高僧", pair_to_code=pair_to_code)
print(token_ids)

[2109, 5352]


In [273]:
encode("我想吃三个西瓜", pair_to_code=pair_to_code)

[7689, 521, 913, 550, 2489]

In [274]:
def decode(tokens, vocab):
    result = list(tokens)
    
    while True:
        i = 0
        new_result = []
        temp_vocab = dict((i, vocab[i]) for i in result if i in vocab)
        if len(temp_vocab) == 0:
            break
        token = max(temp_vocab, key=lambda x: x)
        c1, c2 = temp_vocab[token]
        for t in result:
            if t == token:
                new_result.append(c1)
                new_result.append(c2)
            else:
                new_result.append(t)
        result = new_result
        
    return bytes(result).decode(errors="replace")


def by_token_decode(tokens, vocab):
    results = []
    for token in tokens:
        by_token_str = decode([token], vocab)
        results.append(by_token_str)
    return results
    

In [275]:
for i in range(1000, 2000):
    print(
        decode([i], vocab=vocab),
        end="\n" if i % 30 == 0 else ", ")

皆, 丹, 双, 理, 吃了, 泼, 怎的, 丈, 指, 灯, 破, 排, 场, 顶, 撞, 施, 骂, 箍, �, 应, 楼
盘, �, �, 捉, 只见那, 别, 尊, 起来, 告, 许, 孙行者, 阵, 皇, 全, 平, 修, 右, 雾, 铁棒, 西天,  那, 古, 与我, 容, 遇, 夫, 不住, 器, 曰, 虽
响, 引, 贼, 欲, 御, 哭, 肯, 晚, 认得, 婆, 黑, �, 你这, 天王, 阳, �, 计, 烧, 比, 旁, 其, �, 舍, 离, 房, 觉, 来的, 不要, 元, 堂
 却说, 龙王, 那老, 轮, 皮, 妖怪, 寿, 棍, 彩, 茶, 就是, 师徒, 绳, 果然, 见他, 太子, 东土, 钻, 怒, 如来, 我师父, 银, 台, 偷, 强, 持, 复, 不能, 活, 臣
进去, 凡, 兢, �, 凶, 邪, 消, 有些, 晓, 性命, 卷, 我的, 他的, 孙大圣, �, 杖, 滚, 枪, 老爷, 宗, 恶, 他一, 服, 丢, 《, 》, 按, 讲, 贫, 观看
信, 翻, 摆, 厮, 咒, 神通, 公主, 烟, 幌, 诗, 锦, 整, 奈, 沙僧道, 人家, 药, 腰, 汉, 忍, 唬, 妖魔, 不见, 失, 玉帝, 断, 照, 变作, 岁, 书, 你们
识, 泪, 贤, �, 弟子, 久, 猴王, �, 有个, 得那, 思, 答, 跟, 桃, 饶, �, 教他, �, 哥哥, 跑, �, 造, 结, 崖, 华, 姓, 土地, 玄, 推, 担
遍, 武, 在那, 禅, 题, �下, 该, 问道, 特, �, 道士, 毫, 门外, 师兄, 叉, 一齐, 执, 段, 欢喜, 退, 筋, 钉, 宿, 乃是, �, 都是, 妖王, 那呆子, 壁, 虚
跪, 素, 谁, 败, 群, 威, 在那里, �, 挂, 养, �, 魂, 贫僧, 架, 云头, 行李, 蒙, �, 一个个, 布, 端, 诸, 广, 层, 让, 陛下, 太宗, 争, 索, 模
借, 掌, 怎生, 院, 换, 盖, 凤, 耍, 尘, 则, 波, 及, 美, 乐, 肉, 愿, 跌, 号, 足, 背, �, 闻得, 庄, 多少, �, 金箍, 圣僧, 走了, 空中, 叫做
变做, 嚷, 爷爷, 牙, 老者, 那妖精, 食, 部, 江, 列, 幸, 包, 兵器, 众僧,

In [276]:
for i in range(15000, 15500):
    print(
        decode([i], vocab=vocab),
        end="\n" if i % 30 == 0 else "| ")

平人
蒸了| 唐僧在马上| 饶了| 戏他一| 戏他一戏| 竹杖| 的哭| 还生| 嘴唇| 打死一个| 嫉妒| 金箍儿| 我当时| 打杀他| 西下| 偿命| 遁法| 命也| 骸骨| 迷人| 磨刀| 哭丧棒| 海水| 那见| 藏于| 火灭| 作起| 倒头| 有斋| 把行李
散闷| 离东土| 塔下| 他也曾| 龙蛇| 是个和尚| 连你| 冒冒| 长嘴大耳的和尚| 包儿| 十三年前| 三藏点头| 梦魂| 上我门| 那猪八戒| 摆尾| 长亭| 华盖| 金阶| 昭阳| 阁门大| 阁门大使| 有佛| 国王大喜道| 顿百| 黄袍怪| 深为| 三载| 逢山| 悟能八戒
行路| 有三十六| 我这里有| 筑破| 掣钯| 便砍| 迎来| 难敌| 钻进| 原来是你| 簸| 簸箕| 将公主|  那妖| 沙僧见| 你父王| 变得好| 他能| 本庄| 与他同| 直直| 该有| 又选| 宫娥彩女| 吹弹| 西东| 人讲| 那怪见了| 斟上| 尖尖
单丝| 单丝不| 单丝不线| 单丝不线孤| 单丝不线孤掌| 单丝不线孤掌难| 单丝不线孤掌难鸣| 化龙| 救了师父| 捞上| 入山| 拿来我看| 想我| 不收| 耍子儿| 递了| 白龙马| 乘机| 城去| 一掼| 肯放| 就与他| 古书云| 之话| 行者说| 姻缘| 怎不| 觉道| 炼成| 信他
韬| 破绽| 急收| 一抹| 天师道| 七宿| 被大圣| 上天宫| 飞星| 驿中| 搀住道| 大开东阁| 送出城| 万缘| 佇立| 衲衣| 胆小| 递解| 那樵子| 这里人| 有我哩| 乖巧| 不济事| 以心问心| 径走| 八戒看见| 棺木| 两处| 撒起| 奔上大路
把钉钯| 一半| 又行| 放下钯| 又不会| 搜寻| 本路| 我把你个| 误了大事| 没计| 没计奈何| 且莫忙| 便来| 那来的| 遮日月| 使起| 一往一来| 屎| 没用| 哥哥不要| 沙僧挑担| 虚惊| 众怪道| 等我们| 道者| 你要吃| 驮他| 背在身上| 那山上| 慢走
担山| 莲花洞里| 也只如此| 杯儿| 吊起| 东廊| 两件宝贝| 贴上| 为名| 是孙大圣| 神通广| 当值| 我认得他| 助功| 装得| 如何又| 动脚| 好宝贝| 铜钱| 写个| 合同| 便宜| 走了罢| 假妆| 吃唐僧肉| 一气| 纬| 行者暗想道| 石崖上| 脯
唐僧的肉| 错认| 不答应| 拴住| 厨中| 孙

Turn down the logging level a little bit

In [277]:
import sys
logger.remove()
logger.add(sys.stderr, level="WARNING")

4

In [278]:
token_ids = encode("贫僧来自东土大唐", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['贫僧', '来', '自', '东土大唐']

In [279]:
token_ids = encode("东土大唐", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['东土大唐']

In [280]:
token_ids = encode("悟空", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['悟空']

In [281]:

token_ids = encode("hello", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['h', 'e', 'l', 'l', 'o']

In [284]:

token_ids = encode(text[30000:31000], pair_to_code=pair_to_code)
print("|".join(by_token_decode(token_ids, vocab=vocab)))

好吃|！|”|崩|、|巴|二将|道|：|“|大圣|在|天宫|吃了|仙酒|、|仙肴|是以|椰|酒|不|甚|美|口|。|常言道|：|‘|美|不美|乡|中|水|。|’|”|大圣道|：|“|你们|就是|‘|亲|不|亲|故|乡|人|。|’|我|今早|在|瑶池|中|受用|时|见那|长廊|之下|有许多|瓶|罐|都是那|玉液|琼浆|。|你们都|不曾|尝|着|。|待我|再去|偷|他|几|瓶|回来|你们|各饮|半|杯|一个个|也|长生不老|。|”|众猴|欢喜|不胜|。|大圣|即出|洞门|又|翻|一筋斗|使个隐身法|径至|蟠桃会上|。|进|瑶池|宫阙|只见那|几个|造|酒|、|盘|糟|、|运|水|、|烧火的|还|鼾|睡|未|醒|。|他将|大的|从|左右|胁下|挟|了|两个|两手|提|了|两个|即|拨转云头|回来|会|众猴|在于|洞中|就|做个|“|仙酒|会|”|各饮|了几|杯|快乐|不题|。|

|  |  |却说那|七衣仙女|自|受了|大圣|的|定身法|术|一|周天|方能|解脱|。|各|提|花|篮|回奏|王母|说道|：|“|齐天大圣|使|法术|困住|我等|故此|来迟|。|”|王母|问道|：|“|你等|摘|了多少|蟠桃|？|”|仙女道|：|“|只有|两|篮|小|桃|三|篮|中|桃|。|至|后面|大|桃|半个|也无|想|都是|大圣|偷吃了|。|及|正|寻|间|不期|大圣|走|将出来|行凶|�|�|打|又问|设宴|请谁|。|我等|把|上|会|事|说了一遍|他就|定|住|我等|不知去向|。|只|到如今|才得|醒|解|回来|。|”|

|  |  |王母|闻言即|去|见玉帝|备|陈|前事|。|说不了|又见那|造|酒|的一|班|人|同|仙|官|等|来奏|：|“|不知|甚么人|搅乱|了|‘|蟠桃|大会|’|偷吃了|玉液|琼浆|其|八|珍|百味|亦俱|偷吃了|。|”|又有|四个|大|天师|来|奏上|：|“|太上|道祖|来了|。|”|玉帝|即同|王母|出|迎|。|老君|朝|礼毕道|：|“|老道|宫中|炼|了些|‘|九转|金丹|’|伺候|陛下|做|‘|丹|元|大会|’|不期|被贼|偷去|特|启|陛下|知之|。|”|玉帝|见|奏|悚惧|。|少时|又有|齐天府|仙吏|叩头道|：|“|孙大圣|不|守|执事|自|昨日|出|游|至今|未|转|更不知|去向|。|”|玉帝|又|添|疑|思|。|只见那|赤脚大仙|又|俯囟

In [283]:
decode(encode(text[:1000], pair_to_code), vocab) == text[:1000]

True